# 03 — Layer Inventory

Builds the layer catalog from all discovered OGC services and asset sources.
- Aggregates layer metadata from WMS/WMTS/WFS capabilities
- Adds non-OGC sources (GeoJSON, GeoTIFF, tile URLs)
- Outputs `data/interim/layer_catalog.csv`

In [ ]:
from __future__ import annotations

import pathlib
import pandas as pd
from gdynia_thermal_audit.settings import Settings
from gdynia_thermal_audit.logging_utils import setup_logging
from gdynia_thermal_audit.parser.layer_catalog_builder import build_layer_catalog
from gdynia_thermal_audit.utils.io import ensure_dir, load_csv

setup_logging()
settings = Settings()
INTERIM_DIR = pathlib.Path(settings.data_dir) / 'interim'
ensure_dir(INTERIM_DIR)

In [ ]:
# Load probe results (or demo fallback)
probe_path = INTERIM_DIR / 'probe_results.csv'
if probe_path.exists():
    df_probe = pd.read_csv(probe_path)
else:
    df_probe = pd.read_csv('data/demo/demo_layer_catalog.csv')
    print('Using demo layer catalog')

# Build the layer catalog
sources = df_probe.to_dict(orient='records') if 'url' in df_probe.columns else []
df_catalog = build_layer_catalog(sources)
print(df_catalog.columns.tolist())
df_catalog.to_csv(INTERIM_DIR / 'layer_catalog.csv', index=False)
print(f'{len(df_catalog)} layers catalogued')
df_catalog.head()

## Notes

Inspect `layer_catalog.csv` to identify:
- Layers with `service_type == 'WMS'` suitable for zonal stats (Scenario A)
- Layers with `service_type == 'WFS'` suitable for vector indicator computation (Scenario B)
- Any missing CRS information (requires manual fix before processing)